In [0]:
dbutils.widgets.text("reset_all_data", "false", "Reset Data")
reset_all_data = dbutils.widgets.get("reset_all_data") == "true"

In [0]:
from pyspark.sql.functions import pandas_udf
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.functions import col, udf, length, pandas_udf
import os
import mlflow
import yaml
from typing import Iterator
from mlflow import MlflowClient
mlflow.set_registry_uri('databricks-uc')

# Set up logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logging.getLogger("py4j.java_gateway").setLevel(logging.ERROR)
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)
logging.getLogger('mlflow').setLevel(logging.ERROR) # Disable MLflow warnings
from urllib3.connectionpool import log as urllib3_log
urllib3_log.setLevel(logging.ERROR)

# Workaround for a bug fix that is in progress
mlflow.spark.autolog(disable=True)

import warnings
warnings.filterwarnings("ignore")

In [0]:
if reset_all_data:
  print(f'clearing up schema {config.CATALOG}.{config.SCHEMA}')
  _ = spark.sql(f"DROP DATABASE IF EXISTS `{config.CATALOG}.{config.SCHEMA}` CASCADE")

In [ ]:
def use_and_create_db(CATALOG, SCHEMA, cloud_storage_path = None):
  print(f"USE CATALOG `{CATALOG}`")
  _ = spark.sql(f"USE CATALOG `{CATALOG}`")
  _ = spark.sql(f"""CREATE DATABASE IF NOT EXISTS `{SCHEMA}` """)

def _update_config_catalog(new_catalog):
  """Update the catalog in config.py and the in-memory config object so chain files use the correct catalog at serving time."""
  import re
  config_path = os.path.join(os.getcwd(), 'config.py')
  with open(config_path, 'r') as f:
    content = f.read()
  content = re.sub(r'self\.CATALOG = ".*?"', f'self.CATALOG = "{new_catalog}"', content)
  content = re.sub(r'self\.CATALOG_CACHE = ".*?"', f'self.CATALOG_CACHE = "{new_catalog}"', content)
  with open(config_path, 'w') as f:
    f.write(content)
  # Also update in-memory config
  config.CATALOG = new_catalog
  config.CATALOG_CACHE = new_catalog
  for attr in ['SOURCE_TABLE_FULLNAME', 'EVALUATION_TABLE_FULLNAME', 'VS_INDEX_FULLNAME', 'MODEL_FULLNAME']:
    setattr(config, attr, f"{config.CATALOG}.{config.SCHEMA}.{attr.replace('_FULLNAME','').lower().replace('source_table','databricks_documentation').replace('evaluation_table','eval_databricks_documentation').replace('vs_index','databricks_documentation_vs_index').replace('model','standard_rag_chatbot')}")
  config.SOURCE_TABLE_FULLNAME = f"{config.CATALOG}.{config.SCHEMA}.databricks_documentation"
  config.EVALUATION_TABLE_FULLNAME = f"{config.CATALOG}.{config.SCHEMA}.eval_databricks_documentation"
  config.VS_INDEX_FULLNAME = f"{config.CATALOG}.{config.SCHEMA}.databricks_documentation_vs_index"
  config.MODEL_FULLNAME = f"{config.CATALOG}.{config.SCHEMA}.standard_rag_chatbot"
  config.VS_INDEX_FULLNAME_CACHE = f"{config.CATALOG}.{config.SCHEMA}.cache_vs_index"
  config.VS_METRICS_INDEX_FULLNAME_CACHE = f"{config.CATALOG}.{config.SCHEMA}.metrics"
  config.MODEL_FULLNAME_CACHE = f"{config.CATALOG}.{config.SCHEMA}.rag_chatbot_with_cache"
  print(f"Updated config.py to use catalog '{new_catalog}'")

#If the catalog is defined, we force it to the given value and throw exception if not.
if len(config.CATALOG) > 0:
  current_catalog = spark.sql("SELECT current_catalog()").collect()[0]['current_catalog()']
  if current_catalog != config.CATALOG:
    catalogs = [r['catalog'] for r in spark.sql("SHOW CATALOGS").collect()]
    if config.CATALOG not in catalogs:
      try:
        _ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {config.CATALOG}")
      except Exception as e:
        if "storage root URL" in str(e) or "MANAGED LOCATION" in str(e) or "Default Storage" in str(e):
          # Workspace does not have default storage; fall back to the workspace's existing managed catalog
          managed_catalogs = [r['catalog'] for r in spark.sql("SHOW CATALOGS").collect() 
                              if not r['catalog'].startswith('system') and r['catalog'] != 'samples']
          fallback = [c for c in managed_catalogs if 'shared' not in c.lower() and 'delta' not in c.lower()]
          if fallback:
            fallback_catalog = fallback[0]
            print(f"WARN: Cannot create catalog '{config.CATALOG}'. Falling back to '{fallback_catalog}'.")
            _update_config_catalog(fallback_catalog)
          else:
            raise Exception(f"Cannot create catalog '{config.CATALOG}' and no suitable fallback found. Please create it manually via the workspace UI.")
        else:
          raise e
  use_and_create_db(config.CATALOG, config.SCHEMA)

print(f"using catalog.database `{config.CATALOG}`.`{config.SCHEMA}`")
_ = spark.sql(f"""USE `{config.CATALOG}`.`{config.SCHEMA}`""")

In [0]:
if not spark.catalog.tableExists(config.SOURCE_TABLE_FULLNAME) or spark.table(config.SOURCE_TABLE_FULLNAME).isEmpty() or \
    not spark.catalog.tableExists(config.EVALUATION_TABLE_FULLNAME) or spark.table(config.EVALUATION_TABLE_FULLNAME).isEmpty():
  _ = spark.sql(f'''CREATE TABLE IF NOT EXISTS {config.SOURCE_TABLE_FULLNAME} (
            id BIGINT GENERATED BY DEFAULT AS IDENTITY,
            url STRING,
            content STRING
          ) TBLPROPERTIES (delta.enableChangeDataFeed = true)''')
  (spark.createDataFrame(pd.read_parquet('https://notebooks.databricks.com/demos/dbdemos-dataset/llm/databricks-documentation/databricks_documentation.parquet'))
   .drop('title').write.mode('overwrite').saveAsTable(config.SOURCE_TABLE_FULLNAME))
  (spark.createDataFrame(pd.read_parquet('https://notebooks.databricks.com/demos/dbdemos-dataset/llm/databricks-documentation/databricks_doc_eval_set.parquet'))
   .write.mode('overwrite').saveAsTable(config.EVALUATION_TABLE_FULLNAME))
  # Make sure enableChangeDataFeed is enabled
  _ = spark.sql('ALTER TABLE databricks_documentation SET TBLPROPERTIES (delta.enableChangeDataFeed = true)')